In [2]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import silhouette_score
import warnings
import pickle
warnings.filterwarnings('ignore')


def post_process_predictions_for_maturity(predictions, features, true_labels,
                                          target_mature_clusters=2, min_purity=0.7):
    print("\nPost-processing clusters to improve mature cluster purity...")

    cluster_purities = {}
    for cluster_id in np.unique(predictions):
        cluster_mask = predictions == cluster_id
        if np.sum(cluster_mask) > 0:
            mature_count = np.sum((true_labels == 1) & cluster_mask)
            total_count = np.sum(cluster_mask)
            purity = mature_count / total_count
            cluster_purities[cluster_id] = {
                'purity': purity,
                'mature_count': mature_count,
                'total_count': total_count,
                'features': features[cluster_mask],
                'indices': np.where(cluster_mask)[0]
            }

    mature_candidates = []
    for cluster_id, stats in cluster_purities.items():
        if stats['purity'] >= min_purity:
            mature_candidates.append((cluster_id, stats['purity'], stats['mature_count'], stats['total_count']))

    mature_candidates.sort(key=lambda x: x[1], reverse=True)

    print(f"Current high-purity mature clusters: {len(mature_candidates)}")
    for cluster_id, purity, mature_count, total_count in mature_candidates:
        print(f"  Cluster {cluster_id}: Purity {purity:.1%}, Mature {mature_count}/{total_count}")

    if len(mature_candidates) < target_mature_clusters:
        print(f"Insufficient mature clusters ({len(mature_candidates)} < {target_mature_clusters}), optimizing...")

        near_mature_candidates = []
        for cluster_id, stats in cluster_purities.items():
            if 0.5 <= stats['purity'] < min_purity:
                near_mature_candidates.append((cluster_id, stats['purity'], stats))

        near_mature_candidates.sort(key=lambda x: x[1], reverse=True)

        for cluster_id, purity, stats in near_mature_candidates[:target_mature_clusters - len(mature_candidates)]:
            print(f"  Optimizing cluster {cluster_id} (Current purity: {purity:.1%})...")

            if len(stats['indices']) > 10:
                from sklearn.cluster import AgglomerativeClustering
                sub_kmeans = AgglomerativeClustering(n_clusters=2)
                sub_features = stats['features']
                sub_labels = sub_kmeans.fit_predict(sub_features)

                for sub_cluster in [0, 1]:
                    sub_mask = sub_labels == sub_cluster
                    if np.sum(sub_mask) > 0:
                        sub_mature_count = np.sum((true_labels[stats['indices']] == 1) & sub_mask)
                        sub_total = np.sum(sub_mask)
                        sub_purity = sub_mature_count / sub_total if sub_total > 0 else 0

                        if sub_purity >= min_purity and sub_total >= 20:
                            print(f"    Sub-cluster {sub_cluster}: Purity {sub_purity:.1%}, Samples {sub_total}")

                            for i, idx in enumerate(stats['indices']):
                                if sub_labels[i] == sub_cluster:
                                    pass
                                else:
                                    predictions[idx] = -1

    return predictions


def create_maturity_levels_with_purity_guarantee(predictions, true_labels,
                                                 target_immature=4, target_mature=2,
                                                 min_immature_purity=0.8, min_mature_purity=0.7):
    print(f"\nCreating maturity levels (Target: {target_immature} immature + {target_mature} mature)...")

    cluster_stats = {}
    for cluster_id in np.unique(predictions):
        if cluster_id == -1:
            continue

        cluster_mask = predictions == cluster_id
        if np.sum(cluster_mask) > 0:
            immature_count = np.sum((true_labels == 0) & cluster_mask)
            mature_count = np.sum((true_labels == 1) & cluster_mask)
            total_count = np.sum(cluster_mask)

            immature_ratio = immature_count / total_count
            mature_ratio = mature_count / total_count

            if immature_ratio >= mature_ratio:
                dominant_class = 0
                purity = immature_ratio
            else:
                dominant_class = 1
                purity = mature_ratio

            cluster_stats[cluster_id] = {
                'count': total_count,
                'immature_count': immature_count,
                'mature_count': mature_count,
                'immature_ratio': immature_ratio,
                'mature_ratio': mature_ratio,
                'purity': purity,
                'dominant_class': dominant_class
            }

    immature_clusters = []
    mature_clusters = []

    for cluster_id, stats in cluster_stats.items():
        if stats['dominant_class'] == 0:
            if stats['purity'] >= min_immature_purity:
                immature_clusters.append((cluster_id, stats['purity'], stats['count']))
        else:
            if stats['purity'] >= min_mature_purity:
                mature_clusters.append((cluster_id, stats['purity'], stats['count']))

    immature_clusters.sort(key=lambda x: x[1], reverse=True)
    mature_clusters.sort(key=lambda x: x[1], reverse=True)

    print(f"Available clusters: {len(immature_clusters)} immature, {len(mature_clusters)} mature")

    selected_immature = immature_clusters[:min(target_immature, len(immature_clusters))]
    selected_mature = mature_clusters[:min(target_mature, len(mature_clusters))]

    if len(selected_mature) < target_mature:
        print(f"Insufficient high-purity mature clusters, searching for near-target clusters...")
        all_mature = []
        for cluster_id, stats in cluster_stats.items():
            if stats['dominant_class'] == 1 and stats['purity'] >= 0.6:
                all_mature.append((cluster_id, stats['purity'], stats['count']))

        all_mature.sort(key=lambda x: x[1], reverse=True)
        selected_mature = all_mature[:min(target_mature, len(all_mature))]

    cluster_to_level = {}
    level_descriptions = {}

    for i, (cluster_id, purity, count) in enumerate(selected_immature):
        level = -(i + 1)
        cluster_to_level[cluster_id] = level
        purity_status = "High Purity" if purity >= min_immature_purity else "Medium Purity"
        level_descriptions[level] = f"Immature Level {abs(level)} {purity_status} (Purity: {purity:.1%}, Samples: {count})"

    for i, (cluster_id, purity, count) in enumerate(selected_mature):
        level = i + 1
        cluster_to_level[cluster_id] = level
        purity_status = "High Purity" if purity >= min_mature_purity else "Medium Purity"
        level_descriptions[level] = f"Mature Level {level} {purity_status} (Purity: {purity:.1%}, Samples: {count})"

    maturity_levels = np.array([cluster_to_level.get(label, 0) for label in predictions])

    unique_levels = np.unique(maturity_levels)
    immature_levels = [l for l in unique_levels if l < 0]
    mature_levels = [l for l in unique_levels if l > 0]

    print(f"\nFinal Level Assignment:")
    print(f"  Immature Levels: {len(immature_levels)}")
    for level in sorted(immature_levels):
        print(f"    Level {level}: {level_descriptions[level]}")

    print(f"  Mature Levels: {len(mature_levels)}")
    for level in sorted(mature_levels):
        print(f"    Level {level}: {level_descriptions[level]}")

    return cluster_to_level, level_descriptions, maturity_levels


def add_robust_noise(features, noise_type='gaussian', noise_level=0.1, feature_stats=None):
    np.random.seed(42)
    noisy_features = features.copy()

    if feature_stats is None:
        feature_mean = features.mean()
        feature_std = features.std()
    else:
        feature_mean, feature_std = feature_stats

    if noise_type == 'gaussian':
        noise = np.random.normal(0, feature_std * noise_level, size=features.shape)
        noisy_features += noise

    elif noise_type == 'salt_pepper':
        mask = np.random.choice([0, 1, 2], size=features.shape, p=[1-noise_level, noise_level/2, noise_level/2])
        noisy_features[mask == 1] = features.max()
        noisy_features[mask == 2] = features.min()

    elif noise_type == 'dropout':
        mask = np.random.binomial(1, 1-noise_level, size=features.shape)
        noisy_features *= mask

    elif noise_type == 'uniform':
        noise = np.random.uniform(-feature_std*noise_level, feature_std*noise_level, size=features.shape)
        noisy_features += noise

    return noisy_features


def get_level_statistics(maturity_levels, original_labels, level_descriptions, dataset_name):
    unique_levels = np.unique(maturity_levels)

    immature_levels = [l for l in unique_levels if l < 0]
    mature_levels = [l for l in unique_levels if l > 0]

    print(f"\n{dataset_name} Level Statistics:")
    print("Level\tSamples\tRaw_Immature\tRaw_Mature\tImmature_Ratio\tMature_Ratio\tPurity\tDescription")
    print("-"*120)

    level_stats = []

    for level in sorted(immature_levels, reverse=True):
        level_mask = maturity_levels == level
        level_samples = np.sum(level_mask)

        immature_count = np.sum((original_labels == 0) & level_mask)
        mature_count = np.sum((original_labels == 1) & level_mask)

        immature_ratio = immature_count / level_samples if level_samples > 0 else 0
        mature_ratio = mature_count / level_samples if level_samples > 0 else 0
        purity = immature_ratio

        purity_symbol = "✓" if purity >= 0.8 else "⚠" if purity >= 0.7 else "✗"

        print(f"{purity_symbol} Level {level}\t{level_samples}\t{immature_count}\t{mature_count}\t"
              f"{immature_ratio:>7.1%}\t{mature_ratio:>7.1%}\t{purity:>6.1%}\t{level_descriptions.get(level, '')}")

        level_stats.append({
            'Level': level,
            'Samples': level_samples,
            'Raw_Immature': immature_count,
            'Raw_Mature': mature_count,
            'Immature_Ratio': immature_ratio,
            'Mature_Ratio': mature_ratio,
            'Purity': purity
        })

    for level in sorted(mature_levels):
        level_mask = maturity_levels == level
        level_samples = np.sum(level_mask)

        immature_count = np.sum((original_labels == 0) & level_mask)
        mature_count = np.sum((original_labels == 1) & level_mask)

        immature_ratio = immature_count / level_samples if level_samples > 0 else 0
        mature_ratio = mature_count / level_samples if level_samples > 0 else 0
        purity = mature_ratio

        purity_symbol = "✓" if purity >= 0.7 else "⚠" if purity >= 0.6 else "✗"

        print(f"{purity_symbol} Level {level}\t{level_samples}\t{immature_count}\t{mature_count}\t"
              f"{immature_ratio:>7.1%}\t{mature_ratio:>7.1%}\t{purity:>6.1%}\t{level_descriptions.get(level, '')}")

        level_stats.append({
            'Level': level,
            'Samples': level_samples,
            'Raw_Immature': immature_count,
            'Raw_Mature': mature_count,
            'Immature_Ratio': immature_ratio,
            'Mature_Ratio': mature_ratio,
            'Purity': purity
        })

    unclassified = np.sum(maturity_levels == 0)
    if unclassified > 0:
        print(f"\nUnclassified samples: {unclassified}")

    return pd.DataFrame(level_stats)


def main_direct_maturity_optimization(add_noise=False, noise_type='gaussian', noise_level=0.1):
    print("LDA Maturity Classification System")
    print("="*60)
    print("Goal: Ensure mature level purity ≥70% while maintaining classification structure")
    if add_noise:
        print(f"⚠️ Robustness Validation: {noise_type} noise added to test set (intensity={noise_level})")
    print("="*60)

    print("\n1. Loading feature data...")
    if not os.path.exists('extracted_features_complete.pkl'):
        print("Error: Feature data file not found")
        return None

    features_data = joblib.load('extracted_features_complete.pkl')

    train_features = features_data['train_features']
    test_features = features_data['test_features']
    train_labels = features_data['train_labels']
    test_labels = features_data['test_labels']
    train_paths = features_data['train_paths']
    test_paths = features_data['test_paths']

    print(f"Training features: {train_features.shape}")
    print(f"Test features: {test_features.shape}")
    print(f"Global feature mean: {train_features.mean():.2f}, Global std: {train_features.std():.2f}")

    feature_stats = (train_features.mean(), train_features.std())

    print("\n2. Training LDA model...")

    best_params = {
        'n_clusters': 6,
        'n_immature_clusters': 4,
        'n_mature_clusters': 2,
        'min_mature_purity': 0.7,
        'add_noise': add_noise,
        'noise_type': noise_type if add_noise else 'none',
        'noise_level': noise_level if add_noise else 0.0
    }

    train_features_non_neg = np.abs(train_features)
    lda_model = LatentDirichletAllocation(
        n_components=6,
        random_state=42,
        learning_method='batch'
    )
    topic_dist = lda_model.fit_transform(train_features_non_neg)
    train_predictions = np.argmax(topic_dist, axis=1)

    print(f"LDA clustering completed, 6 clusters generated")

    train_predictions = post_process_predictions_for_maturity(
        train_predictions, train_features, train_labels,
        target_mature_clusters=best_params['n_mature_clusters'],
        min_purity=best_params['min_mature_purity']
    )

    from sklearn.neighbors import KNeighborsClassifier
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(train_features, train_predictions)

    test_predictions_clean = None
    label_change_rate = 0.0
    if add_noise:
        print(f"\n⚠️ Adding {noise_type} noise to test features (relative intensity={noise_level})...")
        test_features_clean = test_features.copy()
        test_features = add_robust_noise(test_features, noise_type, noise_level, feature_stats)

        test_predictions_clean = knn.predict(test_features_clean)
        test_predictions_noisy = knn.predict(test_features)
        label_change_rate = np.mean(test_predictions_clean != test_predictions_noisy)
        print(f"✓ Noise injection complete, label change rate: {label_change_rate:.1%}")

    test_predictions = knn.predict(test_features)

    print("\n3. Creating maturity levels...")
    cluster_to_level, level_descriptions, train_maturity_levels = create_maturity_levels_with_purity_guarantee(
        train_predictions, train_labels,
        target_immature=best_params['n_immature_clusters'],
        target_mature=best_params['n_mature_clusters'],
        min_immature_purity=0.8,
        min_mature_purity=0.7
    )

    test_maturity_levels = np.array([cluster_to_level.get(label, 0) for label in test_predictions])

    train_classified = np.sum(train_maturity_levels != 0)
    test_classified = np.sum(test_maturity_levels != 0)
    train_coverage = train_classified / len(train_maturity_levels)
    test_coverage = test_classified / len(test_maturity_levels)

    print(f"\nClassification Coverage:")
    print(f"  Training set: {train_classified}/{len(train_maturity_levels)} = {train_coverage:.1%}")
    print(f"  Test set: {test_classified}/{len(test_maturity_levels)} = {test_coverage:.1%}")

    print("\n4. Generating detailed statistics...")

    train_stats_df = get_level_statistics(train_maturity_levels, train_labels, level_descriptions, "Training Set")
    test_stats_df = get_level_statistics(test_maturity_levels, test_labels, level_descriptions, "Test Set")

    print("\n5. Saving results...")

    detailed_results = []

    for i in range(len(train_paths)):
        result = {
            'Dataset': 'Training',
            'Image_Name': os.path.basename(train_paths[i]),
            'Original_Label': 'Immature' if train_labels[i] == 0 else 'Mature',
            'Cluster_ID': int(train_predictions[i]) if i < len(train_predictions) else -1,
            'Maturity_Level': int(train_maturity_levels[i]) if train_maturity_levels[i] != 0 else "Unclassified",
            'Level_Description': level_descriptions.get(train_maturity_levels[i], 'Unclassified')
        }
        detailed_results.append(result)

    for i in range(len(test_paths)):
        result = {
            'Dataset': 'Test',
            'Image_Name': os.path.basename(test_paths[i]),
            'Original_Label': 'Immature' if test_labels[i] == 0 else 'Mature',
            'Cluster_ID': int(test_predictions[i]) if i < len(test_predictions) else -1,
            'Maturity_Level': int(test_maturity_levels[i]) if test_maturity_levels[i] != 0 else "Unclassified",
            'Level_Description': level_descriptions.get(test_maturity_levels[i], 'Unclassified')
        }
        detailed_results.append(result)

    detailed_df = pd.DataFrame(detailed_results)

    if add_noise:
        output_file = f'LDA_6Clusters_Maturity_Results_{noise_type}_Noise{noise_level}.xlsx'
        model_file = f'lda_maturity_classifier_{noise_type}_Noise{noise_level}.pkl'
    else:
        output_file = 'LDA_6Clusters_Maturity_Results.xlsx'
        model_file = 'lda_maturity_classifier.pkl'

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        train_stats_df.to_excel(writer, sheet_name='Training_Stats', index=False)
        test_stats_df.to_excel(writer, sheet_name='Test_Stats', index=False)
        detailed_df.to_excel(writer, sheet_name='Detailed_Results', index=False)

        params_df = pd.DataFrame([{'Parameter': k, 'Value': v} for k, v in best_params.items()])
        params_df.to_excel(writer, sheet_name='Parameters', index=False)

        desc_df = pd.DataFrame([{'Level': k, 'Description': v} for k, v in level_descriptions.items()])
        desc_df.to_excel(writer, sheet_name='Level_Descriptions', index=False)

    print(f"\n✓ Results saved to: {output_file}")

    print("\n6. Saving model...")
    lda_data = {
        'best_params': best_params,
        'train_predictions': train_predictions,
        'test_predictions': test_predictions,
        'train_maturity_levels': train_maturity_levels,
        'test_maturity_levels': test_maturity_levels,
        'cluster_to_level': cluster_to_level,
        'level_descriptions': level_descriptions,
        'train_coverage': train_coverage,
        'test_coverage': test_coverage,
        'model': lda_model,
        'knn': knn,
        'feature_stats': feature_stats
    }

    with open(model_file, 'wb') as f:
        pickle.dump(lda_data, f)

    print(f"✓ Model saved to: {model_file}")

    print("\n" + "="*60)
    print("Classification Complete Summary:")
    print("="*60)

    mature_purities = []
    for _, row in train_stats_df.iterrows():
        if row['Level'] > 0:
            mature_purities.append(row['Purity'])

    if mature_purities:
        avg_mature_purity = np.mean(mature_purities)
        print(f"Average mature level purity: {avg_mature_purity:.1%}")

        if len(mature_purities) >= 2:
            print(f"Second mature level (Level 2) purity: {mature_purities[1]:.1%}")

    print(f"Training set coverage: {train_coverage:.1%}")
    print(f"Test set coverage: {test_coverage:.1%}")

    if add_noise and test_predictions_clean is not None:
        print(f"\nRobustness Metrics:")
        print(f"  Label change rate: {label_change_rate:.1%}")
        test_maturity_levels_clean = np.array([cluster_to_level.get(label, 0) for label in test_predictions_clean])
        level_agreement = np.mean(test_maturity_levels == test_maturity_levels_clean)
        print(f"  Maturity level agreement: {level_agreement:.1%}")

    return lda_data


if __name__ == "__main__":
    print("LDA Maturity Classification System")
    print("="*60)
    print("Forced 6 clusters for classification")
    print("="*60)

    lda_data = main_direct_maturity_optimization(add_noise=True, noise_type='gaussian', noise_level=2)

LDA Maturity Classification System
Forced 6 clusters for classification
LDA Maturity Classification System
Goal: Ensure mature level purity ≥70% while maintaining classification structure
⚠️ Robustness Validation: gaussian noise added to test set (intensity=2)

1. Loading feature data...
Training features: (3296, 1024)
Test features: (824, 1024)
Global feature mean: 0.86, Global std: 0.77

2. Training LDA model...
LDA clustering completed, 6 clusters generated

Post-processing clusters to improve mature cluster purity...
Current high-purity mature clusters: 2
  Cluster 4: Purity 97.1%, Mature 545/561
  Cluster 2: Purity 90.3%, Mature 337/373

⚠️ Adding gaussian noise to test features (relative intensity=2)...
✓ Noise injection complete, label change rate: 6.2%

3. Creating maturity levels...

Creating maturity levels (Target: 4 immature + 2 mature)...
Available clusters: 4 immature, 2 mature

Final Level Assignment:
  Immature Levels: 4
    Level -4: Immature Level 4 High Purity (Purit